# LSSO ImageNet-1K launcher

This notebook prepares the exact CUDA runtime required by the current LSSO ImageNet recipe, validates the pinned WebDataset shard layout, launches one detached single-node training job, and provides a refreshable monitor.

It pins the source and runtime ABI deliberately. It reuses a compatible active or local LSSO virtual environment first (set `LSSO_PYTHON` to select one explicitly), and provisions Conda/Torch only when none is available. It never compiles LSSO. Canonical 800-epoch pretraining still requires Apex FusedLAMB; that is checked explicitly and never silently replaced with a different optimizer.

ImageNet remains access-controlled. The notebook uses an authenticated ModelScope download of `timm/imagenet-1k-wds`, validates its pinned `_info.json` manifest, and requires `imagenet1k-train-0000.tar` through `-1023.tar` plus `imagenet1k-validation-00.tar` through `-63.tar`. The manifest is authoritative because the dataset card's validation-shard text is inconsistent. It never expands the data into an ImageFolder tree or uses an unofficial mirror or torrent.

Set `LSSO_IMAGENET_WDS_ROOT` to choose the shard directory. The notebook passes it unchanged as the training `--data-root`.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

REPOSITORY_URL = 'https://github.com/Yang916-yy/LSSO.git'
REPOSITORY_REVISION = 'be4e1c294e1688ce8ef406ad5368e4421c29fd7d'
RUNTIME_URL = (
    'https://github.com/Yang916-yy/LSSO/releases/download/v0.6.0/'
    'lsso_cuda_runtime-0.6.0%2Btorch2110cu128-py3-none-linux_x86_64.whl'
)
RUNTIME_SHA256 = 'f8f939e1264a71fa7c38e8c506f6feb04b75007162f5d081f3b87c0d9370a4c1'
TARGET_TORCH = '2.11.0'
TARGET_CUDA = '12.8'
ENV_NAME = 'lsso-imagenet-torch211-cu128'
EXISTING_PYTHON = os.environ.get('LSSO_PYTHON')  # Optional explicit compatible interpreter.

cwd = Path.cwd().resolve()
if not (cwd / 'pyproject.toml').is_file() and (cwd.parent / 'pyproject.toml').is_file():
    cwd = cwd.parent
REPOSITORY_DIR = Path(os.environ.get('LSSO_REPOSITORY_DIR', str(cwd if (cwd / 'pyproject.toml').is_file() else Path.home() / 'LSSO'))).expanduser()
# --data-root receives this ModelScope WebDataset root directly; it is not an ImageFolder tree.
DATA_ROOT = Path(os.environ.get('LSSO_IMAGENET_WDS_ROOT', str(Path.home() / 'data' / 'imagenet-1k-wds'))).expanduser()
RUN_ROOT = Path(os.environ.get('LSSO_IMAGENET_RUN_ROOT', str(Path.home() / 'runs' / 'lsso-imagenet'))).expanduser()
CONTROL_ROOT = Path(os.environ.get('LSSO_IMAGENET_CONTROL_ROOT', str(Path.home() / '.cache' / 'lsso-imagenet'))).expanduser()
CACHE_ROOT = Path(os.environ.get('LSSO_IMAGENET_CACHE_ROOT', str(Path.home() / '.cache' / 'lsso-imagenet' / 'artifacts'))).expanduser()

MODELSCOPE_DATASET = 'timm/imagenet-1k-wds'
MODELSCOPE_REVISION = 'master'
MODELSCOPE_TOKEN = os.environ.get('MODELSCOPE_API_TOKEN')
MODELSCOPE_LOGIN = True  # Set False only when a valid ModelScope session already exists.
DOWNLOAD_FROM_MODELSCOPE = True
ALLOW_MINIFORGE_BOOTSTRAP = True
UPDATE_EXISTING_CLONE = False

TIER = 'small'
PHASE = 'pretrain'  # one of: pretrain, finetune_224
RUN_NAME = f'{TIER}-{PHASE}'
INITIAL_CHECKPOINT: Path | None = None  # required for a new B/L finetune_224 run
PHYSICAL_BATCH_SIZE: int | None = None  # Set a valid per-GPU batch, or leave None for the documented default.
TRAIN_WORKERS: int | None = None  # None uses the repository default (10 per rank).
VAL_WORKERS: int | None = None    # None uses the repository default (4 per rank).
NPROC_PER_NODE: int | None = None       # None uses all visible GPUs
CUDA_VISIBLE_DEVICES: str | None = None
INSTALL_APEX_FOR_PRETRAIN = True

def show_command(command: list[str]) -> None:
    print('$ ' + ' '.join(shlex.quote(part) for part in command))

import shlex

def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None, capture: bool = False) -> subprocess.CompletedProcess[str]:
    show_command(command)
    return subprocess.run(command, cwd=cwd, env=env, check=True, text=True, capture_output=capture)

def output(command: list[str], *, env: dict[str, str] | None = None) -> str:
    return run(command, env=env, capture=True).stdout.strip()

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def version_tuple(value: str) -> tuple[int, ...]:
    return tuple(int(part) for part in value.split('.') if part.isdigit())

if platform.system() != 'Linux' or platform.machine() not in {'x86_64', 'AMD64'}:
    raise RuntimeError('the published runtime requires Linux x86_64')
glibc_version = platform.libc_ver()[1]
glibc_display = glibc_version or 'unknown'
if version_tuple(glibc_version) < (2, 31):
    raise RuntimeError(f'glibc >= 2.31 is required, found {glibc_display}')
run(['nvidia-smi', '-L'])
for directory in (RUN_ROOT, CONTROL_ROOT, CACHE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
def torch_probe(python: Path) -> dict[str, object] | None:
    code = '''
import json
try:
    import torch
    result = {
        'python': __import__('sys').version.split()[0],
        'torch': torch.__version__.split('+')[0],
        'cuda': torch.version.cuda,
        'abi': bool(torch.compiled_with_cxx11_abi()),
        'cuda_available': bool(torch.cuda.is_available()),
    }
except Exception as error:
    result = {'error': repr(error)}
print(json.dumps(result))
'''
    completed = subprocess.run([str(python), '-c', code], text=True, capture_output=True)
    if completed.returncode != 0:
        return None
    return json.loads(completed.stdout)

def matches_runtime(probe: dict[str, object] | None) -> bool:
    if not probe or 'error' in probe:
        return False
    return (
        version_tuple(str(probe['python'])) >= (3, 10)
        and version_tuple(str(probe['python'])) < (3, 13)
        and probe['torch'] == TARGET_TORCH
        and probe['cuda'] == TARGET_CUDA
        and probe['abi'] is True
        and probe['cuda_available'] is True
    )

def find_conda() -> Path | None:
    candidates = [os.environ.get('CONDA_EXE'), shutil.which('conda')]
    candidates.append(str(Path.home() / '.local' / 'miniforge3' / 'bin' / 'conda'))
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate)
    return None

def bootstrap_miniforge() -> Path:
    prefix = Path.home() / '.local' / 'miniforge3'
    conda = prefix / 'bin' / 'conda'
    if conda.is_file():
        return conda
    if not ALLOW_MINIFORGE_BOOTSTRAP:
        raise RuntimeError('no Conda was found; install Conda or set ALLOW_MINIFORGE_BOOTSTRAP = True')
    installer = CACHE_ROOT / 'Miniforge3-Linux-x86_64.sh'
    print('bootstrapping Miniforge under', prefix)
    urllib.request.urlretrieve(
        'https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh',
        installer,
    )
    run(['bash', str(installer), '-b', '-p', str(prefix)])
    if not conda.is_file():
        raise RuntimeError('Miniforge bootstrap did not create conda')
    return conda

def named_conda_prefix(conda: Path, name: str) -> Path | None:
    listing = json.loads(output([str(conda), 'env', 'list', '--json']))
    for prefix in listing.get('envs', []):
        candidate = Path(prefix)
        if candidate.name == name:
            return candidate
    return None

def find_compatible_python() -> Path | None:
    candidates = [
        Path(EXISTING_PYTHON).expanduser() if EXISTING_PYTHON else None,
        Path(sys.executable),
        REPOSITORY_DIR / '.venv' / 'bin' / 'python',
    ]
    candidates.extend(sorted(Path.home().glob('LSSO*/.venv/bin/python')))
    seen: set[Path] = set()
    for candidate in candidates:
        if candidate is None or not candidate.is_file():
            continue
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if matches_runtime(torch_probe(candidate)):
            return candidate
    return None

compatible_python = find_compatible_python()
current_probe = torch_probe(Path(sys.executable))
CONDA_EXE: Path | None = None
CONDA_ENV_PREFIX: Path | None = None
if compatible_python is not None:
    TARGET_PYTHON = compatible_python
    print('reusing compatible Python:', TARGET_PYTHON)
else:
    print('active Python is not compatible:', current_probe)
    CONDA_EXE = find_conda() or bootstrap_miniforge()
    CONDA_ENV_PREFIX = named_conda_prefix(CONDA_EXE, ENV_NAME)
    if CONDA_ENV_PREFIX is None:
        run([str(CONDA_EXE), 'create', '-y', '-n', ENV_NAME, 'python=3.11', 'pip'])
        CONDA_ENV_PREFIX = named_conda_prefix(CONDA_EXE, ENV_NAME)
    if CONDA_ENV_PREFIX is None:
        raise RuntimeError(f'Conda did not create {ENV_NAME}')
    TARGET_PYTHON = CONDA_ENV_PREFIX / 'bin' / 'python'
    existing_probe = torch_probe(TARGET_PYTHON)
    if existing_probe and 'error' not in existing_probe and not matches_runtime(existing_probe):
        raise RuntimeError(
            f'existing {ENV_NAME} is incompatible: {existing_probe}; choose a new ENV_NAME rather than deleting it'
        )

TARGET_ENV = os.environ.copy()
TARGET_ENV['PATH'] = str(TARGET_PYTHON.parent) + os.pathsep + TARGET_ENV.get('PATH', '')
if CUDA_VISIBLE_DEVICES is not None:
    TARGET_ENV['CUDA_VISIBLE_DEVICES'] = CUDA_VISIBLE_DEVICES
print('target Python:', TARGET_PYTHON)


In [ ]:
probe = torch_probe(TARGET_PYTHON)
if not matches_runtime(probe):
    run([str(TARGET_PYTHON), '-m', 'pip', 'install', '--upgrade', 'pip'], env=TARGET_ENV)
    run(
        [
            str(TARGET_PYTHON), '-m', 'pip', 'install',
            f'torch=={TARGET_TORCH}+cu128', 'torchvision==0.26.0+cu128',
            '--index-url', 'https://download.pytorch.org/whl/cu128',
        ],
        env=TARGET_ENV,
    )
    probe = torch_probe(TARGET_PYTHON)
if not matches_runtime(probe):
    raise RuntimeError(f'failed to provision the required torch runtime: {probe}')

run(
    [
        str(TARGET_PYTHON), '-m', 'pip', 'install', '--upgrade',
        'timm>=1.0.16,<2', 'webdataset>=1,<2', 'pillow>=10', 'numpy>=1.26', 'packaging>=24',
    ],
    env=TARGET_ENV,
)

if not REPOSITORY_DIR.exists():
    run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY_DIR)])
    run(['git', 'checkout', '--detach', REPOSITORY_REVISION], cwd=REPOSITORY_DIR)
elif not (REPOSITORY_DIR / '.git').is_dir():
    raise RuntimeError(f'{REPOSITORY_DIR} exists but is not a Git checkout')
else:
    head = output(['git', '-C', str(REPOSITORY_DIR), 'rev-parse', 'HEAD'])
    run(['git', '-C', str(REPOSITORY_DIR), 'fetch', '--tags', 'origin'])
    expected = output(['git', '-C', str(REPOSITORY_DIR), 'rev-parse', f'{REPOSITORY_REVISION}^{{commit}}'])
    if head != expected:
        if not UPDATE_EXISTING_CLONE:
            raise RuntimeError(
                f'{REPOSITORY_DIR} is at {head[:12]}, expected {expected[:12]} from {REPOSITORY_REVISION}; '
                'set UPDATE_EXISTING_CLONE = True only for a clean disposable clone'
            )
        status = output(['git', '-C', str(REPOSITORY_DIR), 'status', '--porcelain'])
        if status:
            raise RuntimeError('refusing to change a dirty repository')
        run(['git', '-C', str(REPOSITORY_DIR), 'checkout', '--detach', expected])

run([str(TARGET_PYTHON), '-m', 'pip', 'install', '--no-deps', '-e', str(REPOSITORY_DIR)], env=TARGET_ENV)

runtime_wheel = CACHE_ROOT / RUNTIME_URL.rsplit('/', 1)[1].replace('%2B', '+')
if not runtime_wheel.is_file() or sha256_file(runtime_wheel) != RUNTIME_SHA256:
    print('downloading the pinned precompiled LSSO runtime')
    urllib.request.urlretrieve(RUNTIME_URL, runtime_wheel)
actual_digest = sha256_file(runtime_wheel)
if actual_digest != RUNTIME_SHA256:
    raise RuntimeError(f'runtime wheel SHA-256 mismatch: {actual_digest}')
run([str(TARGET_PYTHON), '-m', 'pip', 'install', '--upgrade', '--force-reinstall', '--no-deps', str(runtime_wheel)], env=TARGET_ENV)

gpu_info = json.loads(output([str(TARGET_PYTHON), '-c', 'import json, torch; print(json.dumps(dict(count=torch.cuda.device_count(), capability=torch.cuda.get_device_capability(0))))'], env=TARGET_ENV))
if int(gpu_info['count']) < 1:
    raise RuntimeError('the selected environment cannot see a CUDA GPU')
verification = '''
import torch
from lsso import LSSO, LSSOConfig
from lsso.ball import cuda
device = torch.device('cuda')
cuda.load(device=device)
assert cuda.is_available()
layer = LSSO(LSSOConfig(dim=64, num_heads=1, rank=16)).to(device)
x = torch.randn(2, 17, 64, device=device, dtype=torch.float16, requires_grad=True)
y = layer(x, implementation='cuda')
y.float().square().mean().backward()
assert torch.isfinite(y).all() and torch.isfinite(x.grad).all()
print({'cuda_available': cuda.is_available(), 'shape': tuple(y.shape)})
'''
run([str(TARGET_PYTHON), '-c', verification], env=TARGET_ENV)


In [ ]:
def validate_webdataset_root(root: Path) -> tuple[dict[str, object] | None, str]:
    # Execute through TARGET_PYTHON so this notebook and the trainer share one manifest contract.
    validation = '''
import json
import sys
from pathlib import Path
from experiments.imagenet import load_imagenet_webdataset_manifest

manifest = load_imagenet_webdataset_manifest(Path(sys.argv[1]))
print(json.dumps({
    'manifest_sha256': manifest.sha256,
    'train_shards': len(manifest.train.paths),
    'validation_shards': len(manifest.validation.paths),
    'train_samples': manifest.train.num_samples,
    'validation_samples': manifest.validation.num_samples,
    'first_train_shard': manifest.train.paths[0].name,
    'last_train_shard': manifest.train.paths[-1].name,
    'first_validation_shard': manifest.validation.paths[0].name,
    'last_validation_shard': manifest.validation.paths[-1].name,
}, sort_keys=True))
'''
    completed = subprocess.run(
        [str(TARGET_PYTHON), '-c', validation, str(root)],
        cwd=REPOSITORY_DIR,
        env=TARGET_ENV,
        text=True,
        capture_output=True,
    )
    if completed.returncode != 0:
        detail = (completed.stderr or completed.stdout).strip()
        return None, detail[-2000:] or 'the ImageNet WebDataset validator failed'
    try:
        return json.loads(completed.stdout), 'verified'
    except json.JSONDecodeError:
        return None, f'validator produced invalid JSON: {completed.stdout[-1000:]}'

def modelscope_binary() -> Path:
    run([str(TARGET_PYTHON), '-m', 'pip', 'install', '--upgrade', 'modelscope>=1,<2'], env=TARGET_ENV)
    binary = TARGET_PYTHON.parent / 'modelscope'
    if not binary.is_file():
        raise RuntimeError(f'ModelScope CLI was not installed at {binary}')
    return binary

def login_to_modelscope(modelscope: Path) -> None:
    if not MODELSCOPE_LOGIN:
        return
    from getpass import getpass
    token = MODELSCOPE_TOKEN or getpass('ModelScope access token: ')
    if not token:
        raise RuntimeError('a ModelScope access token is required for authenticated ImageNet download')
    login = subprocess.run(
        [str(modelscope), 'login', '--token', token], env=TARGET_ENV, text=True, capture_output=True
    )
    if login.returncode != 0:
        raise RuntimeError(login.stderr[-2000:] or login.stdout[-2000:])
    print('ModelScope login succeeded.')

def download_from_modelscope(root: Path) -> None:
    modelscope = modelscope_binary()
    login_to_modelscope(modelscope)
    root.mkdir(parents=True, exist_ok=True)
    command = [str(modelscope), 'download', MODELSCOPE_DATASET, '--repo-type', 'dataset', '--revision', MODELSCOPE_REVISION, '--local-dir', str(root)]
    print('downloading authenticated ModelScope WebDataset shards into', root)
    run(command, env=TARGET_ENV)

ready, status = validate_webdataset_root(DATA_ROOT)
if ready is None and DOWNLOAD_FROM_MODELSCOPE:
    print('WebDataset layout is incomplete:', status)
    download_from_modelscope(DATA_ROOT)
    ready, status = validate_webdataset_root(DATA_ROOT)
if ready is None:
    raise FileNotFoundError(
        'ImageNet-1K WebDataset is not ready. Set LSSO_IMAGENET_WDS_ROOT to a directory containing '
        'the authenticated ModelScope timm/imagenet-1k-wds tar shards, or enable DOWNLOAD_FROM_MODELSCOPE. '
        f'Validator: {status}'
    )
print({'webdataset_root': str(DATA_ROOT), **ready})


In [ ]:
def apex_available() -> bool:
    check = 'from apex.optimizers import FusedLAMB; print(FusedLAMB.__name__)'
    return subprocess.run([str(TARGET_PYTHON), '-c', check], env=TARGET_ENV).returncode == 0

if PHASE == 'pretrain' and not apex_available():
    if not INSTALL_APEX_FOR_PRETRAIN:
        raise RuntimeError('canonical pretraining requires Apex FusedLAMB; no optimizer fallback is enabled')
    nvcc = shutil.which('nvcc', path=TARGET_ENV['PATH'])
    if nvcc is None and CONDA_EXE is not None and CONDA_ENV_PREFIX is not None:
        run([str(CONDA_EXE), 'install', '-y', '-p', str(CONDA_ENV_PREFIX), '-c', 'nvidia', 'cuda-nvcc=12.8'])
        nvcc = shutil.which('nvcc', path=TARGET_ENV['PATH'])
    if nvcc is None:
        raise RuntimeError('Apex needs CUDA 12.8 nvcc; install it in the target Conda environment and rerun this cell')
    nvcc_version = output([nvcc, '--version'], env=TARGET_ENV)
    if 'release 12.8' not in nvcc_version:
        raise RuntimeError(f'Apex must be built with CUDA 12.8 nvcc, found: {nvcc_version}')
    apex_env = TARGET_ENV | {
        'CUDA_HOME': str(Path(nvcc).resolve().parents[1]),
        'APEX_CPP_EXT': '1',
        'APEX_CUDA_EXT': '1',
        'APEX_PARALLEL_BUILD': str(min(8, os.cpu_count() or 1)),
    }
    run([str(TARGET_PYTHON), '-m', 'pip', 'install', '--no-build-isolation', '--no-cache-dir', 'ninja', 'git+https://github.com/NVIDIA/apex.git'], env=apex_env)
if PHASE == 'pretrain' and not apex_available():
    raise RuntimeError('Apex FusedLAMB remains unavailable; canonical pretraining cannot start')
print('optimizer prerequisite is ready')

def visible_gpu_count() -> int:
    return int(output([str(TARGET_PYTHON), '-c', 'import torch; print(torch.cuda.device_count())'], env=TARGET_ENV))

def batching_defaults(tier: str, phase: str) -> tuple[int, int, int]:
    if phase == 'finetune_224':
        return 512, 64, 64
    if tier in {'small', 'base'}:
        return 2048, 256, 512
    return 2048, 64, 128

world_size = NPROC_PER_NODE or visible_gpu_count()
effective_batch, group_size, preferred_batch = batching_defaults(TIER, PHASE)
if PHYSICAL_BATCH_SIZE is None:
    candidates = [preferred_batch, effective_batch // world_size, group_size]
    PHYSICAL_BATCH_SIZE = next(
        (candidate for candidate in candidates if candidate >= group_size and candidate % group_size == 0 and effective_batch % (world_size * candidate) == 0),
        None,
    )
if PHYSICAL_BATCH_SIZE is None or PHYSICAL_BATCH_SIZE % group_size or effective_batch % (world_size * PHYSICAL_BATCH_SIZE):
    raise ValueError(
        f'world_size={world_size}, physical_batch={PHYSICAL_BATCH_SIZE} cannot satisfy the virtual-batch contract '
        f'for effective_batch={effective_batch}, group_size={group_size}'
    )
grad_accum = effective_batch // (world_size * PHYSICAL_BATCH_SIZE)
print({'world_size': world_size, 'physical_batch': PHYSICAL_BATCH_SIZE, 'grad_accum': grad_accum, 'effective_batch': effective_batch, 'group_size': group_size})

output_dir = RUN_ROOT / RUN_NAME
checkpoint = output_dir / 'checkpoint_last.pt'
control_path = CONTROL_ROOT / f'{RUN_NAME}.json'
lock_path = CONTROL_ROOT / f'{RUN_NAME}.lock'
log_path = CONTROL_ROOT / f'{RUN_NAME}-{int(time.time())}.log'
if control_path.is_file():
    existing = json.loads(control_path.read_text())
    try:
        os.kill(int(existing['pid']), 0)
    except OSError:
        control_path.unlink()
    else:
        existing_pid = existing['pid']
        raise RuntimeError(f'a job for {RUN_NAME} is already running as PID {existing_pid}')

command = [
    str(TARGET_PYTHON), '-m', 'torch.distributed.run', '--standalone',
    '--nproc_per_node', str(world_size), 'experiments/train_imagenet.py',
    '--tier', TIER, '--phase', PHASE, '--data-root', str(DATA_ROOT),
    '--output', str(output_dir), '--batch-size', str(PHYSICAL_BATCH_SIZE),
    '--grad-accum', str(grad_accum),
]
for flag, workers in (('--train-workers', TRAIN_WORKERS), ('--val-workers', VAL_WORKERS)):
    if workers is not None:
        if workers < 0:
            raise ValueError(f'{flag} must be non-negative')
        command += [flag, str(workers)]
if checkpoint.is_file():
    command += ['--resume', str(checkpoint)]
elif PHASE == 'finetune_224':
    if INITIAL_CHECKPOINT is None or not Path(INITIAL_CHECKPOINT).is_file():
        raise FileNotFoundError('a new B/L finetune_224 run requires INITIAL_CHECKPOINT')
    command += ['--init-checkpoint', str(INITIAL_CHECKPOINT)]
elif output_dir.exists() and any(output_dir.iterdir()):
    raise RuntimeError(f'{output_dir} is nonempty without checkpoint_last.pt; choose a new RUN_NAME')

launch = ['flock', '-n', str(lock_path), *command]
launch_env = TARGET_ENV.copy()
if CUDA_VISIBLE_DEVICES is not None:
    launch_env['CUDA_VISIBLE_DEVICES'] = CUDA_VISIBLE_DEVICES
show_command(launch)
with log_path.open('ab') as log_handle:
    process = subprocess.Popen(
        launch, cwd=REPOSITORY_DIR, env=launch_env, stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True
    )
time.sleep(1)
if process.poll() is not None:
    print(log_path.read_text(errors='replace')[-4000:])
    raise RuntimeError(f'launcher exited immediately with code {process.returncode}')
control_path.write_text(json.dumps({'pid': process.pid, 'command': command, 'output': str(output_dir), 'log': str(log_path)}, indent=2) + '\n')
print(f'launched PID {process.pid}; rerun the monitor cell to refresh status')


In [ ]:
from IPython.display import clear_output, display

def tail(path: Path, lines: int = 40) -> str:
    if not path.is_file():
        return '(no log yet)'
    with path.open('rb') as handle:
        handle.seek(max(0, path.stat().st_size - 16384))
        return '\n'.join(handle.read().decode(errors='replace').splitlines()[-lines:])

def refresh_monitor() -> None:
    clear_output(wait=True)
    if not control_path.is_file():
        print('no notebook-controlled job is registered for', RUN_NAME)
        return
    control = json.loads(control_path.read_text())
    pid = int(control['pid'])
    try:
        os.kill(pid, 0)
        state = 'running'
    except OSError:
        state = 'stopped'
    print({'state': state, 'pid': pid, 'output': control['output'], 'log': control['log']})
    metrics_path = Path(control['output']) / 'metrics.jsonl'
    if metrics_path.is_file():
        records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
        if records:
            print('latest metric:', json.dumps(records[-1], indent=2))
    smi = subprocess.run(
        ['nvidia-smi', '--query-gpu=index,name,utilization.gpu,memory.used,memory.total', '--format=csv,noheader'],
        text=True, capture_output=True,
    )
    if smi.returncode == 0:
        print('GPU:', smi.stdout.strip())
    print('\nlast log lines:\n' + tail(Path(control['log'])))

try:
    import ipywidgets as widgets
    refresh_button = widgets.Button(description='Refresh training status', icon='refresh')
    refresh_button.on_click(lambda _: refresh_monitor())
    display(refresh_button)
except ImportError:
    print('ipywidgets is unavailable; rerun this cell to refresh')
refresh_monitor()


In [ ]:
# Set STOP_NOTEBOOK_JOB = True only when intentionally stopping this notebook-managed process group.
STOP_NOTEBOOK_JOB = False
if STOP_NOTEBOOK_JOB:
    if not control_path.is_file():
        raise RuntimeError('no notebook-controlled job is registered')
    import signal
    control = json.loads(control_path.read_text())
    os.killpg(int(control['pid']), signal.SIGTERM)
    print('sent SIGTERM to process group', control['pid'])
